# PS-10 Stage III — change detection pipeline

One notebook runs the whole project (HANDOFF.md section 10). Every cell calls a script or a `scd` function, is idempotent, and is executable with papermill (`make notebook STAGE=all`). Set `STAGES` to run a subset, e.g. `"0,1,2"`.

In [ ]:
# parameters
CONFIG = "configs/default.yaml"
OVERRIDES = []                 # e.g. ["train.epochs=2"]
RUN_NAME = "v1_seed42"
STAGES = "all"                 # or "0,1,2" etc.
CHIPS_VERSION = 1
INFER_PAIR = ""                # "<aoi_id>/<pair_id>" for section 13; default: first val pair
APPEND_CSV = ""                # optional CSV of AOIs to append in section 1
ABLATIONS = []                 # e.g. ["A5_softmax5", "A8_hard15"] for section 14

## 0. Setup

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from scd import log, status
from scd.config import load

os.chdir(Path.cwd() if (Path.cwd() / "scd").exists() else Path.cwd().parent)
cfg = load(CONFIG, OVERRIDES)
log.banner(cfg)

_stages = None if str(STAGES) == "all" else {int(s) for s in str(STAGES).split(",")}


def enabled(stage: int) -> bool:
    return _stages is None or stage in _stages


def run(stage: int, script: str, *args: str, config: str | None = None) -> None:
    """Run a scripts/*.py CLI as a subprocess with this notebook's --config/--set."""
    if not enabled(stage):
        print(f"[skip] stage {stage}: {script}")
        return
    cmd = [sys.executable, f"scripts/{script}", "--config", config or CONFIG]
    for o in OVERRIDES:
        cmd += ["--set", o]
    cmd += list(args)
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def mark(stage: int, name: str, outputs=()) -> None:
    if enabled(stage):
        status.write(cfg, f"{stage:02d}_{name}", [str(o) for o in outputs])
        print(f"status: stage {stage} {name} recorded")


print("data root:", cfg.data_root, "| exists:", cfg.data_root.exists())

In [ ]:
mark(0, "setup", [])  # STATUS -> runs/pipeline_status.json

## 1. Registry

In [ ]:
from scd.imagery import registry

reg_path = registry.registry_path(cfg)
if enabled(1) and not reg_path.exists():
    run(1, "00_init_registry.py")
if APPEND_CSV:
    run(1, "00_init_registry.py", "--append-csv", APPEND_CSV)
aois = registry.load(reg_path) if reg_path.exists() else None
if aois is not None:
    display(aois.groupby(["stratum", "role"]).size().unstack(fill_value=0))
    display(aois.groupby(["stratum", "target_type"]).size().unstack(fill_value=0))
    errors = registry.validate(aois)
    print("registry valid" if not errors else "\n".join(errors))

In [ ]:
import folium

if aois is not None:
    colours = {"A": "purple", "B": "orange", "C": "green", "D": "blue", "E": "cadetblue", "F": "lightgray"}
    m = folium.Map(location=[float(aois.centre_lat.mean()), float(aois.centre_lon.mean())], zoom_start=4)
    for _, r in aois.iterrows():
        folium.CircleMarker([r.centre_lat, r.centre_lon], radius=4, color=colours.get(r.stratum, "black"),
                            popup=f"{r.aoi_id} ({r.role})").add_to(m)
    display(m)

In [ ]:
mark(1, "registry", [reg_path])  # STATUS -> runs/pipeline_status.json

## 2. Scenes

In [ ]:
run(2, "01_fetch_scenes.py", "--query-only")
scenes_path = cfg.registry_dir / "scenes.parquet"
if scenes_path.exists():
    scenes = pd.read_parquet(scenes_path)
    display(scenes.groupby("aoi_id").size().describe())
    display(scenes.head())

In [ ]:
run(2, "01_fetch_scenes.py", "--download")
pairs_path = cfg.registry_dir / "pairs.parquet"
if pairs_path.exists():
    pairs = pd.read_parquet(pairs_path)
    display(pairs["status"].value_counts())

In [ ]:
mark(2, "scenes", [cfg.registry_dir / 'scenes.parquet', cfg.registry_dir / 'pairs.parquet'])  # STATUS -> runs/pipeline_status.json

## 3. Pairs

In [ ]:
run(3, "02_preprocess_pairs.py")
metas = [json.loads(p.read_text()) | {"path": str(p.parent)} for p in cfg.pairs_dir.glob("*/*/meta.json")]
meta_df = pd.DataFrame(metas)
if len(meta_df):
    display(meta_df[["aoi_id", "pair_id", "valid_fraction", "dx", "dy", "same_orbit", "season_match"]].describe(include="all"))
    ax = meta_df[["dx", "dy"]].plot.hist(bins=30, alpha=0.6, title="co-registration offsets (px)")

In [ ]:
import random
from IPython.display import Image

if len(meta_df):
    for stratum, grp in meta_df.groupby(meta_df["aoi_id"].str[0]):
        p = Path(random.Random(0).choice(list(grp["path"]))) / "quicklook.png"
        if p.exists():
            print(stratum, p)
            display(Image(filename=str(p), width=900))

In [ ]:
if enabled(3):
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_preprocess_symmetry.py"], check=False)

In [ ]:
mark(3, "pairs", [cfg.pairs_dir])  # STATUS -> runs/pipeline_status.json

## 4. Candidates

In [ ]:
run(4, "03_mine_candidates.py", "--sources", "gem,cbra,obt,io_lulc,osm_history")
from scd.labels import mining

if enabled(4):
    display(mining.candidate_counts(cfg))

In [ ]:
mark(4, "candidates", [cfg.candidates_dir])  # STATUS -> runs/pipeline_status.json

## 5. Review

In [ ]:
run(5, "04_export_for_review.py")
from scd.labels import verification

display(verification.pending(cfg))

In [ ]:
# after humans finish in QGIS / the review app:
run(5, "05_import_reviewed.py")
if enabled(5):
    display(verification.agreement_table(cfg))

In [ ]:
mark(5, "review", [cfg.reviewed_dir])  # STATUS -> runs/pipeline_status.json

## 6. Labels

In [ ]:
run(6, "06_rasterize.py")
from scd.labels import rasterize

if enabled(6):
    display(rasterize.area_table(cfg))

In [ ]:
mark(6, "labels", [cfg.labels_dir])  # STATUS -> runs/pipeline_status.json

## 7. Chips

In [ ]:
run(7, "07_build_chips.py", "--version", str(CHIPS_VERSION), "--sampler-report")
from scd.data import manifest

man_path = cfg.chips_dir(CHIPS_VERSION) / "manifest.parquet"
if man_path.exists():
    man = manifest.load(man_path)
    display(manifest.summarise(man))
    print("class weights:", manifest.class_weights(man, cfg.train.loss.class_weight_power))

In [ ]:
mark(7, "chips", [man_path])  # STATUS -> runs/pipeline_status.json

## 8. Object library

In [ ]:
run(8, "08_build_object_library.py", "--version", str(CHIPS_VERSION))
from scd.labels import synthetic

lib_dir = cfg.objects_dir(CHIPS_VERSION)
if lib_dir.exists():
    lib = synthetic.ObjectLibrary.load(lib_dir)
    print(lib.counts())
    synthetic.show_gallery(lib, n=8)
    synthetic.show_paste_demo(cfg, lib, man_path)

In [ ]:
mark(8, "objects", [lib_dir])  # STATUS -> runs/pipeline_status.json

## 9. Baseline (PCCD flicker)

In [ ]:
run(9, "12_flicker_baseline.py", "--strata", "A,B,C,D,E,F", "--n-pairs", "10")
flick = cfg.runs_dir / "flicker.csv"
if flick.exists():
    display(pd.read_csv(flick))

In [ ]:
mark(9, "baseline", [flick])  # STATUS -> runs/pipeline_status.json

## 10. Train

In [ ]:
if enabled(10):
    cmd = [sys.executable, "scripts/09_train.py", "--config", CONFIG, "--run", RUN_NAME, "--chips-version", str(CHIPS_VERSION)]
    for o in OVERRIDES:
        cmd += ["--set", o]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd)
    metrics_csv = cfg.run_dir(RUN_NAME) / "logs" / "metrics.csv"
    import time
    while proc.poll() is None:
        time.sleep(60)
        if metrics_csv.exists():
            print(pd.read_csv(metrics_csv).tail(1).to_string())
    assert proc.returncode == 0, "training failed"

In [ ]:
metrics_csv = cfg.run_dir(RUN_NAME) / "logs" / "metrics.csv"
if metrics_csv.exists():
    m = pd.read_csv(metrics_csv)
    cols = [c for c in ["val/leaderboard_proxy", "val/weighted_iou", "val/robust_swap", "val/robust_identity",
                        "val/robust_synthetic", "val/robust_near"] if c in m]
    m.dropna(subset=cols, how="all").set_index("epoch")[cols].plot(subplots=True, figsize=(8, 10))

In [ ]:
mark(10, "train", [cfg.run_dir(RUN_NAME)])  # STATUS -> runs/pipeline_status.json

## 11. Calibrate

In [ ]:
run(11, "10_calibrate.py", "--run", RUN_NAME, "--chips-version", str(CHIPS_VERSION))
calib_path = cfg.run_dir(RUN_NAME) / "calibration.json"
if calib_path.exists():
    print(json.dumps(json.loads(calib_path.read_text()), indent=2))

In [ ]:
mark(11, "calibrate", [calib_path])  # STATUS -> runs/pipeline_status.json

## 12. Evaluate

In [ ]:
run(12, "13_evaluate.py", "--run", RUN_NAME, "--chips-version", str(CHIPS_VERSION))
from scd.eval import evaluate

if enabled(12):
    evaluate.show(cfg, RUN_NAME)

In [ ]:
mark(12, "evaluate", [cfg.run_dir(RUN_NAME) / 'metrics'])  # STATUS -> runs/pipeline_status.json

## 13. Infer one pair (+ swap and identity checks)

In [ ]:
from scd.infer import pipeline

pair_dir = pipeline.pick_pair(cfg, INFER_PAIR, chips_version=CHIPS_VERSION)
out_dir = cfg.outputs_dir / f"nb_{RUN_NAME}_{pair_dir.parent.name}_{pair_dir.name}"
run(13, "11_infer.py", "--run", RUN_NAME, "--t1", str(pair_dir / "t1.tif"), "--t2", str(pair_dir / "t2.tif"), "--out", str(out_dir))
if enabled(13):
    pipeline.show_result(out_dir)
    print(pipeline.invariance_checks(cfg, RUN_NAME, pair_dir, out_dir.parent / (out_dir.name + "_checks")))

In [ ]:
mark(13, "infer", [out_dir])  # STATUS -> runs/pipeline_status.json

## 14. Ablations

In [ ]:
ablation_ids = sorted(p.stem for p in Path("configs/ablations").glob("*.yaml"))
print("available:", ablation_ids)
for a in ABLATIONS:
    run(14, "14_ablation.py", "--id", a, "--chips-version", str(CHIPS_VERSION))
abl_csv = cfg.runs_dir / "ablations.csv"
if abl_csv.exists():
    display(pd.read_csv(abl_csv))

In [ ]:
mark(14, "ablations", [abl_csv])  # STATUS -> runs/pipeline_status.json

## 15. Fine-tune (Stage III)

In [ ]:
run(15, "15_finetune.py", "--run", f"{RUN_NAME}_ft", config="configs/finetune.yaml")
from scd.train import train

if enabled(15):
    display(train.forgetting_table(cfg, RUN_NAME, f"{RUN_NAME}_ft"))

In [ ]:
mark(15, "finetune", [cfg.run_dir(f'{RUN_NAME}_ft')])  # STATUS -> runs/pipeline_status.json

## 16. Report

In [ ]:
run(16, "16_report.py", "--run", RUN_NAME)
rep = cfg.run_dir(RUN_NAME) / "report.md"
if rep.exists():
    display(Markdown(rep.read_text()))

In [ ]:
mark(16, "report", [rep])  # STATUS -> runs/pipeline_status.json